# EMD3 cell-state conversion and selection

Run all cells with the default settings to reproduce Figure 5. The population model tests which measurements distinguish conversion from selective proliferation or death using synthetic data. The Boolean model compares outcomes across starting states. Installation instructions are in the [README](README.md).

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import importlib.metadata
import platform
import subprocess
from time import perf_counter
from IPython.display import Image, Markdown, display

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents)
             if (p / "src" / "emd3_simulation" / "run.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter in the EMD3 directory.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from emd3_simulation import FIGURE_FORMATS, FIGURE_STEM

## Run settings

Use `reference` for the manuscript results. The `quick` setting uses 40 synthetic datasets per mechanism and schedule and skips the parameter-sensitivity ensemble. Choose a new `RUN_NAME` to keep an exploratory run. Reusing a name replaces its generated files.

In [ ]:
RUN_MODE = "reference"  # "reference" or "quick"
RUN_NAME = "reproduced"
N_DATASETS = 120
N_ENSEMBLE = 60
MAKE_FIGURE = True
EXPORT_MANUSCRIPT_TABLES = True

if RUN_MODE not in {"reference", "quick"}:
    raise ValueError("RUN_MODE must be reference or quick.")
if not isinstance(RUN_NAME, str) or not RUN_NAME or Path(RUN_NAME).name != RUN_NAME or RUN_NAME in {".", ".."}:
    raise ValueError("RUN_NAME must be one folder name.")
if RUN_MODE == "quick":
    N_DATASETS, N_ENSEMBLE = 40, 0
    if RUN_NAME == "reproduced":
        RUN_NAME = "quick"
if not isinstance(N_DATASETS, int) or isinstance(N_DATASETS, bool) or N_DATASETS < 1:
    raise ValueError("N_DATASETS must be a positive integer.")
if not isinstance(N_ENSEMBLE, int) or isinstance(N_ENSEMBLE, bool) or N_ENSEMBLE < 0:
    raise ValueError("N_ENSEMBLE must be a non-negative integer.")
REFERENCE_RUN = N_DATASETS == 120 and N_ENSEMBLE == 60
OUTDIR = ROOT / "outputs" / RUN_NAME
OUTDIR.mkdir(parents=True, exist_ok=True)
print(f"{N_DATASETS} synthetic datasets per mechanism and schedule, {N_ENSEMBLE} sensitivity draws")
print("Output folder:", OUTDIR)

## Run the analysis

Conversion and selection are matched to the same stem-like fraction on day 14. The analysis fits marker fractions on short and extended schedules. On the extended schedule it also compares model selection using fractions alone or fractions with absolute counts. The reference run also exports the model tables.

In [ ]:
summary_path = OUTDIR / "emd3_simulation_summary.json"
table_path = OUTDIR / "emd3_manuscript_tables.json"
metadata_path = OUTDIR / "emd3_notebook_run_metadata.json"
log_path = OUTDIR / "emd3_run.log"
figure_paths = [OUTDIR / f"{FIGURE_STEM}.{ext}" for ext in FIGURE_FORMATS]
caption_path = OUTDIR / f"{FIGURE_STEM}_legend.md"
for path in (summary_path, table_path, metadata_path, caption_path, *figure_paths):
    path.unlink(missing_ok=True)

command = [sys.executable, "-u", "-m", "emd3_simulation.run",
           "--n-draws", str(N_DATASETS), "--n-ensemble", str(N_ENSEMBLE),
           "--outdir", str(OUTDIR)]
if not MAKE_FIGURE:
    command.append("--no-figure")

print("Progress log:", log_path, flush=True)
started = perf_counter()
with log_path.open("w", encoding="utf-8") as log:
    completed = subprocess.run(command, cwd=ROOT / "src", stdout=log, stderr=subprocess.STDOUT)
if not summary_path.is_file():
    raise RuntimeError(f"The analysis produced no summary. See {log_path}")
summary = json.loads(summary_path.read_text())
checks = summary["checks"]
if completed.returncode:
    failed = [name for name, passed in checks.items() if not passed]
    if REFERENCE_RUN or completed.returncode != 1 or not failed:
        raise RuntimeError(f"The analysis failed. See {log_path}")
    print("Model checks that failed in this run:", ", ".join(failed))

if EXPORT_MANUSCRIPT_TABLES and REFERENCE_RUN:
    with log_path.open("a", encoding="utf-8") as log:
        subprocess.run([sys.executable, "-m", "emd3_simulation.manuscript_tables",
                        "--outdir", str(OUTDIR)], cwd=ROOT / "src", stdout=log,
                       stderr=subprocess.STDOUT, check=True)

source_files = sorted((ROOT / "src" / "emd3_simulation").rglob("*.py"))
run_metadata = {
    "run_mode": RUN_MODE, "n_datasets": N_DATASETS, "n_ensemble": N_ENSEMBLE,
    "reference_configuration": REFERENCE_RUN, "workflow_exit_code": completed.returncode,
    "figure_requested": MAKE_FIGURE, "python": platform.python_version(),
    "packages": {name: importlib.metadata.version(name)
                 for name in ("numpy", "scipy", "matplotlib", "nbclient")},
    "source_sha256": {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest()
                      for p in source_files},
    "interpretation": "Population model fits use synthetic observations and assigned baseline rates."
}
metadata_path.write_text(json.dumps(run_metadata, indent=2) + "\n", encoding="utf-8")
print(f"Completed in {perf_counter() - started:.1f} seconds. {sum(checks.values())}/{len(checks)} model checks passed.")

## Figure

In [ ]:
if MAKE_FIGURE:
    display(Image(filename=str(OUTDIR / f"{FIGURE_STEM}.png"), width=1100))
    display(Markdown(caption_path.read_text(encoding="utf-8")))

## Conversion-rate estimates

The table shows the percentage of synthetic datasets in which the profile interval for increased conversion excludes zero. Compare the fit that fixes the difference in net growth rates with the fit that estimates it. Selection and differential death both have zero exposure-induced conversion in the data-generating model.

In [ ]:
rows = [
    "| Schedule | Generating mechanism | Growth difference fixed | Growth difference estimated |",
    "| --- | --- | ---: | ---: |",
]
for schedule, mechanisms in summary["false_positive_rates"].items():
    for mechanism, fixed in mechanisms.items():
        estimated = summary["false_positive_rates_both"][schedule][mechanism]
        rows.append(f"| {schedule} | {mechanism} | {fixed['ci_excl']:.1%} | {estimated['ci_excl']:.1%} |")
display(Markdown("\n".join(rows)))

## Reproducibility checks

Verify the synthetic dataset count and the number of asynchronous update sequences per starting state. The reference run also checks completion of all 60 sensitivity draws and all 17 model checks.

In [ ]:
settings = summary["provenance"]
assert settings["n_datasets_per_truth_and_design"] == N_DATASETS
assert settings["n_ensemble_draws"] == N_ENSEMBLE
assert {row["n_traj"] for row in summary["boolean"].values()} == {settings["boolean_trajectories_per_start"]}
assert settings["boolean_trajectories_per_start"] == 32
ensemble = summary.get("ensemble")
if N_ENSEMBLE:
    assert ensemble["n_requested"] == N_ENSEMBLE
    print(f"Completed sensitivity draws: {ensemble['n_ok']}/{N_ENSEMBLE}")
else:
    assert ensemble is None
if REFERENCE_RUN:
    assert ensemble["n_ok"] == N_ENSEMBLE
    assert len(checks) == 17 and all(checks.values())

outputs = [summary_path, log_path, metadata_path]
if MAKE_FIGURE:
    outputs.extend([*figure_paths, caption_path])
if EXPORT_MANUSCRIPT_TABLES and REFERENCE_RUN:
    outputs.append(table_path)
for path in outputs:
    assert path.is_file() and path.stat().st_size > 0, path
print(f"Run configuration verified. Results are in {OUTDIR.relative_to(ROOT)}/")